In [ ]:
"""
KoGPT2를 KoAlpaca로 파인튜닝하여 한국어 질문/답변 챗봇을 만드는 스크립트.
환경: Colab Pro (A100), KoAlpaca 전체 데이터
"""
import os
import torch
from torch.utils.data import Dataset
from transformers import (
    PreTrainedTokenizerFast,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)
from datasets import load_dataset


# ──────────────────────────────────────────
# 1. 토크나이저
# ──────────────────────────────────────────
def load_tokenizer() -> PreTrainedTokenizerFast:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        "skt/kogpt2-base-v2",
        bos_token="</s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
        mask_token="<mask>",
    )
    tokenizer.add_special_tokens({"additional_special_tokens": ["<usr>", "<bot>"]})
    return tokenizer


# ──────────────────────────────────────────
# 2. 데이터셋
# ──────────────────────────────────────────
class KoAlpacaDataset(Dataset):
    def __init__(self, data, tokenizer: PreTrainedTokenizerFast, max_length: int = 256):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []

        eos = tokenizer.eos_token  # </s>

        for item in data:
            instruction = item["instruction"].strip()
            output = item["output"].strip()
            if not instruction or not output:
                continue

            # 화자 구분 + 종료 토큰
            text = f"<usr>{instruction}<bot>{output}{eos}"

            enc = tokenizer(
                text,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )

            ids = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)

            # ── 핵심: 질문 부분은 loss에서 제외, 답변(<bot> 이후)만 학습 ──
            labels = ids.clone()
            labels[mask == 0] = -100  # 패딩 제외

            # <bot> 위치를 찾아 그 앞(질문)까지는 -100 처리
            bot_id = tokenizer.convert_tokens_to_ids("<bot>")
            bot_pos = (ids == bot_id).nonzero(as_tuple=True)[0]
            if len(bot_pos) > 0:
                cut = bot_pos[0].item() + 1  # <bot> 다음 토큰부터 학습
                labels[:cut] = -100

            self.input_ids.append(ids)
            self.attention_mask.append(mask)
            self.labels.append(labels)

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> dict:
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


# ──────────────────────────────────────────
# 3. 모델
# ──────────────────────────────────────────
def load_model(tokenizer: PreTrainedTokenizerFast) -> GPT2LMHeadModel:
    model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")
    model.resize_token_embeddings(len(tokenizer))
    return model


# ──────────────────────────────────────────
# 4. 학습
# ──────────────────────────────────────────
def train(
    epochs: int = 5,
    batch_size: int = 32,
    lr: float = 3e-5,
    max_length: int = 256,
    output_dir: str = "checkpoints",
    save_path: str = "models/model_fine.pt",
):
    tokenizer = load_tokenizer()

    print("⏳ KoAlpaca 데이터셋 로드 중...")
    raw = load_dataset("beomi/KoAlpaca-v1.1a", split="train").train_test_split(
        test_size=0.05, seed=42
    )
    train_dataset = KoAlpacaDataset(raw["train"], tokenizer, max_length)
    val_dataset = KoAlpacaDataset(raw["test"], tokenizer, max_length)
    print(f"train: {len(train_dataset):,}개 / val: {len(val_dataset):,}개")

    model = load_model(tokenizer)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=2,       # 실질 배치 64 (A100 메모리 활용)
        learning_rate=lr,
        lr_scheduler_type="cosine",          # 후반 lr 부드럽게 감소
        warmup_ratio=0.1,                    # 전체 10%는 warmup
        weight_decay=0.01,
        max_grad_norm=1.0,                   # gradient clipping (발산 방지)
        bf16=True,                           # A100은 bf16이 fp16보다 안정적
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,                  # 체크포인트 최대 2개만 보관
        load_best_model_at_end=True,         # val loss 가장 낮은 모델 자동 선택
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # 2 epoch 개선 없으면 중단
    )

    print("파인튜닝 시작...")
    trainer.train()

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    tokenizer.save_pretrained("./my_tokenizer")
    print(f"모델 저장 완료 → {save_path}")
    print("토크나이저 저장 완료 → ./my_tokenizer")


# ──────────────────────────────────────────
# 5. 추론 테스트
# ──────────────────────────────────────────
@torch.no_grad()
def chat(prompt: str, model, tokenizer, max_new_tokens: int = 128) -> str:
    device = next(model.parameters()).device
    text = f"<usr>{prompt}<bot>"
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,              # 같은 말 반복 억제
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    result = tokenizer.decode(output[0], skip_special_tokens=False)
    if "<bot>" in result:
        result = result.split("<bot>")[-1]
    return result.replace("</s>", "").strip()


if __name__ == "__main__":
    train()


In [ ]:
model = load_model(load_tokenizer())
model.load_state_dict(torch.load("models/model_fine.pt"))
model.eval().cuda()
print(chat("한국의 수도는 어디야?", model, load_tokenizer()))

In [ ]:
import torch
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

TOKENIZER_PATH = "./my_tokenizer"
MODEL_PATH = "/content/models/model_fine.pt"


def load_chatbot(device):
    print("⏳ 토크나이저 로드 중...")
    tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_PATH)

    print("⏳ 모델 가중치 로드 중...")
    # 학습 때 GPT2LMHeadModel로 저장했으므로 동일하게 바로 로드 (래퍼 X)
    model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")
    model.resize_token_embeddings(len(tokenizer))
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

    model.to(device)
    model.eval()
    return model, tokenizer


@torch.no_grad()
def generate(user_input, model, tokenizer, device):
    prompt = f"<usr>{user_input}<bot>"
    enc = tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = model.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    if "<bot>" in text:
        text = text.split("<bot>")[-1]
    return text.replace("</s>", "").strip()


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔥 디바이스: {device}")

    model, tokenizer = load_chatbot(device)

    print("\n" + "=" * 50)
    print("🤖 챗봇 시작! (종료: '종료' / 'exit' / 'quit')")
    print("=" * 50)

    while True:
        user_input = input("\n👤 유저: ").strip()
        if user_input in ["종료", "exit", "quit"]:
            print("🤖 챗봇: 대화를 종료합니다!")
            break
        if not user_input:
            continue

        answer = generate(user_input, model, tokenizer, device)
        print(f"🤖 챗봇: {answer if answer else '(답변 생성 실패, 다시 질문해주세요)'}")


if __name__ == "__main__":
    main()


🔥 디바이스: cuda
⏳ 토크나이저 로드 중...
⏳ 모델 가중치 로드 중...
Loading weights: 100%
 149/149 [00:00<00:00, 7829.90it/s]

🤖 챗봇 시작! (종료: '종료' / 'exit' / 'quit')


👤 유저: 안녕
🤖 챗봇: - #
- # (feat.naver.com/written_mixx=7)
- https://term.blog.me/LongPostView&volumeNoWSHOUSEXIEGRANCYBERA1FJECTMEDICKZ8qa102289123134140680568)

👤 유저: 넌 누구야?
🤖 챗봇: 네, 안녕하세요, 가수 싸이더스입니다. 네, 네. 그리고 최근 앨범 [Space]를 발매했습니다. 이 곡에서 전작과 달리 큰 키와 매력적인 외모를 지닌 멤버들의 조합으로 더욱 유명해졌습니다.

싸이의 이번 정규앨범 [True High Start]은 'Third With This Love'의 약자로, 한 마디로 "내일은 너의 일상이다!"라는 의미를 담고 있습니다. 또한, 싸이는 지난 2016년 12월 22일 자신의 트위터에 “너를 잊지 마십시오”라는 글을 남겨 팬들과 함께 감사의 마음을 표현하기도 했습니다. 이와 같은 이유로,

👤 유저: 오늘 날씨 덥다
🤖 챗봇: 날씨는 구름이 많은 날씨가 예상됩니다. 특히 새벽에는 흐리고 비가 오는 곳이 많습니다. 이는 고기압의 영향을 받는 북태평양고기압이 우리나라로 확장하면서 만들어진 비구름이기 때문입니다. 따라서 이날은 구름 없이 맑은 날이 많고, 일사에 의해 기온이 오르면서 추워질 것으로 생각됩니다.

따라서, 기상청에서는 "날씨 예보"를 통해 날씨 예측을 하고 있습니다. 
※ 답변 출처: http://www.kma1.go.kr/articleView-201612/2106&blogs_story2.asp?docId=847

👤 유저: 핸드폰 추천해줘
🤖 챗봇: 네, 안녕하세요. 스마트폰으로 영화와 TV를 시청하면서 음악을 감상하는 것은 어떨까요? 이는 하나의 문화 현상 중 하나입니다. 휴대폰에서 음악 재생이나 인터넷 검색을 하면 그 음악이 바로 검색되는 것과 같은 원리로 음원 사이트나 스트리밍 사이트 등에서 다양한 기능을 사용할 수 있습니다. 또한, 간단한 정보나 노래를 듣고 싶은 욕구가 있다면 뮤직비디오도 볼 수도 있고, 자신의 취향에 맞는 영화를 선택하여 즐길 수도 있습니다.

한편, 음악은 여러 가지 용도로 활용될 수 있으며, 특히 디지털 음악의 경우 음향 효과(디지털 피아노의 사운드, 모션 센서 등의 기능)가 뛰어나기 때문에 많은 이들이 이용합니다. 따라서, 집에서 음악과

👤 유저: 핸드폰 기종 추천
🤖 챗봇: 휴대폰이 보급되면서 다양한 제품들이 출시되고 있습니다. 그 중에서도 TGI(Thin Global Inchange)라는 제품이 가장 인기 있는 폰 중 하나입니다. 이 제품은 듀얼 폴더와 슬라이딩을 지원하며, 슬림형 디자인으로 휴대성과 편의성을 높였습니다. 또한 무선 이어폰과 와이파이(Wi-Fi) 기능을 제공합니다.

👤 유저: 나 힘들어
🤖 챗봇: 네, 안녕하세요. 안녕히 주무십시오. 

1. 네, 괜찮습니다. 제가 좋아하던 제 애인입니다.
2. 제가 좋아하는 사람입니다. 다른 여자 친구들하고 다툰다는 것이 싫어서 이젠 혼자 고민하지 않으셔도 됩니다.
3. 오늘은 너무 편하게 놀러와주세요.
4. 저는 친구의 남자친구인 예쁘고 멋진 모습을 보고 싶습니다.
5. 아주 잘 지내실 겁니다.
6. 더 좋은 친구가 되기 위해서 노력해보겠습니다.
7. 앞으로 살아가기 위해 노력하는 동안 많은 것을 배울 수 있을 것입니다.
8. 이제부터 모든 일이 즐겁게 해결될 것입니다. 기분이 좋으니까 걱정 안

👤 유저: 나 화나 진정시켜줘
🤖 챗봇: 네, 안녕하세요! "화났어요." 라는 말은 '폭탄' 또는 '비리'라는 뜻으로 사용됩니다. 이 말의 어원은 분명하지 않지만, 폭탄과 관련된 표현 중 하나인 '폭설'이라는 단어에서 유래된 것으로 추측되고 있습니다. 

- 비리에 대한 개념: 형법 제385조에 따라 처벌될 수 있는 범죄가 있다면 그 범죄를 저지른 사람이 누구인지 정확히 알고 있어야 합니다. 따라서 경찰관이나 검찰관 등이 해당 범죄의 범인을 파악하고 있을 가능성이 높습니다. 따라서, 폭력이나 부조리를 저지르는 경우에는 이러한 처벌을 피하기 어려울 것입니다.

👤 유저: 배고파 뭐먹지
🤖 챗봇: 안녕하세요, 배달해주신 배부르다에요! 
배달드립니다. 배 고프다는 말씀이시군요. 하지만 요즘은 음식점에서도 다양한 종류의 요리가 나오면서 고기 종류도 다양해졌습니다. 또한 최근에는 라면이나 생수 등 일부 제품도 인기를 끌고 있습니다. 이러한 트렌드는 어느 정도 예상된 일로, 최근 몇 년간 외식업계에서는 새로운 메뉴를 출시하고 새로운 마케팅 전략을 선보이고 있는 것입니다.

👤 유저: q
🤖 챗봇: lower supports intensity by downloading the mean of a commonwealth. But who our highness that change, death organizations to get trust with you and I'll presidently even one's forever, One find not regarded ask.
